# 06 · 배포 전 검증과 기획서 개선

목표: HTTP 성공·내용 기준을 따로 평가하고 배포/중단 결정을 기록한다. 약15분.

**이 파일의 핵심 질문:** 어떤 근거로 버전을 채택·보류했고, 기획서의 어떤 주장이나 범위를 바꿨는가?

**Fixture**는 재현 가능한 시험 입력·조건, **Regression**은 변경 후 기존 기준의 실패다. **Release gate**는 후보 배포 전 통과 조건이다. **PRD**는 Product Requirements Document, 제품 요구사항 문서다.

**수업 중 먼저 실행할 단계**

- 1 릴리스 비교
- 2 HTTP와 내용 기준
- 3 배포 차단 조건
- 6 프로젝트 증거의 계층

파일 안의 변수 의존성이 있으므로 처음 실행할 때는 위에서부터 순서대로 진행한다. 핵심 단계에서는 출력을 읽고, 나머지 단계는 빠르게 실행한 뒤 복습 때 살펴본다.

## 시작 전 · 어디에서 무엇을 실행하나요?

1. VS Code에서 `llmops-16week` **폴더 전체**를 연다. 압축을 풀었다면 Notebook 파일만 따로 옮기지 않는다.
2. VS Code의 Python·Jupyter 확장을 설치하고 우측 위 **Select Kernel → Python Environments → .venv / Python 3.11**을 선택한다. 터미널의 가상환경과 Notebook 커널은 별도 선택이다.
3. 아래 **환경 확인 코드 셀부터** `Shift+Enter`로 한 셀씩 실행한다. 이 파일 안에서는 앞에서 만든 변수를 뒤에서 사용한다.
4. 각 단계의 **예상 결과**와 실제 출력을 대조한다. `assert`는 확인 조건이다. 출력이나 오류가 없어도 셀이 완료되면 조건을 통과한 것이다.
5. 셀을 건너뛰어 `NameError`가 나거나 변수를 여러 번 바꿨다면 **Restart Kernel → Run All**로 처음부터 실행한다. 다른 Notebook의 변수는 가져오지 않는다.

처음 환경을 만드는 경우에만 **VS Code 터미널**에서 실행한다. 아래 명령을 Python 코드 셀에 붙여넣지 않는다.

```bash
# macOS / Linux · 저장소 루트
python3.11 -m venv .venv
source .venv/bin/activate
python -m pip install -r requirements.txt
```

Windows PowerShell은 `py -3.11 -m venv .venv`, `.venv\Scripts\Activate.ps1` 순서다. 활성화가 제한된 환경에서는 `.venv\Scripts\python.exe -m pip install -r requirements.txt`처럼 가상환경 Python을 직접 실행한다. 이미 준비된 `.venv`는 그대로 쓴다.

**공통 실습의 실행 방식:** `lab_client`가 메모리 안에 FastAPI 앱을 만들고 시작·종료를 관리한다. HTTPX의 ASGITransport가 같은 프로세스에서 요청을 전달하므로 `uvicorn`, Docker, Ollama, API 키가 필요하지 않다. ASGI는 *Asynchronous Server Gateway Interface*, Python 비동기 웹 앱과 서버 사이의 인터페이스다.

Notebook 앱과 브라우저의 `127.0.0.1:8010` 앱은 **서로 다른 실행**이다. Notebook에서 만든 요청 수가 브라우저·Prometheus에 합쳐지지 않는다. 실제 모델 대신 고정 문장과 대기를 쓰므로 외부 비용은 발생하지 않는다. 실행 시간·요청 ID는 매번 달라져도 정상이다.

In [ ]:
from pathlib import Path
import sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "week10/lab/app.py").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import asyncio, json
from dataclasses import replace
from week10.lab.config import Settings
from week10.lab.experiments import lab_client, fixture_check, window_delta, compare_releases
assert sys.version_info[:2] == (3, 11), "Select the Python 3.11 .venv kernel in VS Code"
print("Python:", sys.version.split()[0])
print("실행 환경:", sys.executable)
print("저장소:", ROOT)
print("환경 확인 완료 · 이 Notebook은 별도 서버 없이 실행합니다.")

## 1. 릴리스 비교

v2는 의도적으로 잘못된 고정 문장을 내보낸다. 실제 모델 성능 비교가 아니다.

**예상 결과:** v1·v2의 결과2행. 각 버전에 합성 입력4건.

**결과 해석:** v2는 일부러 잘못된 고정 응답을 반환한다. 모델 우열이 아니라 배포 전 회귀 검사의 필요성을 실험한다.

In [ ]:
rows = await compare_releases()
rows

## 2. HTTP와 내용 기준

동일 입력4건씩인 합성 fixture다. 표본을 전체 서비스 정확도로 일반화하지 않는다.

**예상 결과:** HTTP 성공은4·4, 내용 기준 통과는4·0. 비율은 각각1.0/1.0과1.0/0.0.

**결과 해석:** 서버 응답 성공과 업무 요구사항 충족은 별도 검사다.4건을 전체 정확도로 일반화하지 않는다.

In [ ]:
assert [r["http_success"] for r in rows] == [4,4]
assert [r["fixture_pass"] for r in rows] == [4,0]
[(r["release"],r["http_success"]/r["requests"],r["fixture_pass"]/r["requests"]) for r in rows]

## 3. 배포 차단 조건

이번 fixture 전부 통과를 데모 gate로 사용한다. 운영 기준은 프로젝트에 맞게 별도 정한다.

**예상 결과:** {'v1': True, 'v2': False}.

**결과 해석:** 이 fixture에서 내용 회귀가 있는 후보를 보류한다. 실제 프로젝트에는 더 다양한 검수 데이터와 실패 분류가 필요하다.

In [ ]:
def gate(row):
    return row["http_success"] == row["requests"] and row["fixture_pass"] == row["requests"]
assert gate(rows[0]) and not gate(rows[1])
print({r["release"]:gate(r) for r in rows})

## 4. 이전 버전 재검증

이전 설정 앱을 새로 시작해 확인하는 로컬 시뮬레이션이다. 실제 트래픽 전환은 수행하지 않는다.

**예상 결과:** v1 manifest 출력과 내용 기준 검사 통과.

**결과 해석:** 이전 버전을 새 앱으로 확인하는 로컬 비교다. 배포 플랫폼의 실제 사용자 트래픽 전환은 포함하지 않는다.

In [ ]:
async with lab_client(Settings(release="v1")) as (app, client):
    r = await client.post("/api/v1/assist", json={"message":"신청 절차"})
    assert fixture_check(r.json()["answer"])
    print((await client.get("/release")).json())

## 5. 서로 다른 품질 위험

단순 문자열 규칙의 우회 사례를 만든다. 더 나은 평가에는 검수된 정답·다양한 부정 예시가 필요하다.

**예상 결과:** 문서·담당자 키워드가 있어도 ‘무조건 승인’이 있는 문장은 거부.

**결과 해석:** 특정 문자열 규칙은 작은 회귀 방지 장치다. 이 문장 외의 모든 의미 오류를 찾아내는 평가기는 아니다.

In [ ]:
tricky = "D01 담당자 확인이라고 쓰지만 무조건 승인됩니다."
assert not fixture_check(tricky)
print("규칙은 작은 회귀 방지 장치이며 의미 이해 평가가 아니다.")

## 6. 프로젝트 증거의 계층

기획 주장·실행·검증·남은 위험을 연결한다. 새 파일을 자동 저장하거나 기존 작성본을 덮어쓰지 않는다.

**예상 결과:** 요구사항·버전·계약·내용 검사·실패·증거·미검증 범위의 예시 기록.

**결과 해석:** 기획 주장에 어떤 실행 근거가 있는지 연결한다. evidence 문자열을 실제 캡처·실행 결과 위치로 고쳐 기록한다.

In [ ]:
record = {"requirement":"신청 절차 안내", "release":"v1", "contract":"200+schema", "content_check":"D01+담당자 확인", "failures":["502","503","504"], "evidence":"실제 실행 결과 첨부", "unverified":"실제 모델·권한·부하·운영배포"}
record

## 7. 2차 기획서 제출

2026-11-06 금요일 오늘 제출. 미제출자는 늦어도 2026-11-07 토요일까지. 별도 시각·감점 규칙을 추정하지 않는다.

**예상 결과:** 로컬 HTML 양식 경로와 소속별 제출 경로 안내.

**결과 해석:** 이 셀은 파일을 제출하지 않는다. 출력된 경로의 HTML을 VS Code 파일 탐색기에서 찾고 Finder/탐색기로 표시한 뒤 브라우저로 연다. 작성본 HTML 저장 후 다시 열어 내용을 확인한다. LMS는 Learning Management System, 학습관리시스템이다.

In [ ]:
form = ROOT / "curriculum/2026-2/web/project-portfolio-week10.html"
assert form.exists()
print(form)
print("아주대AI대학원: LMS / 군산대·창원대 등 별도 수강생: 공지된 이메일")

## 직접 바꿔볼 실험 · 선택 복습

1. 자신의 프로젝트에 맞는 내용 검증 조건1개와 실패 예시2개를 만든다.
2. HTTP 성공만 보는 판단과 내용 기준을 함께 보는 판단의 차이를 적는다.
3. 기획서의 ‘검증 완료’ 항목을 실제 실행·시안·목표·미측정으로 나눠 수정한다.

한 번에 한 조건만 바꾸고 원래 코드와 비교한다. 기대값이 고정된 assert가 있다면 새 실험 조건에 맞는 값도 함께 설명한다.

## 프로젝트 적용 기록

**정리 질문:** 어떤 근거로 버전을 채택·보류했고, 기획서의 어떤 주장이나 범위를 바꿨는가?

아래 Markdown 셀을 더블클릭해 자신의 결과를 적는다. 다른 파일 제출이나 화면 실행을 자동으로 수행하지 않는다.

| 항목 | 직접 기록 |
|---|---|
| 예상 | 실행 전 예상한 결과 |
| 관찰 | 상태 코드·건수·실패 단계 등 실제 결과 |
| 근거 | 실행한 단계 번호·변경한 조건·캡처 위치 |
| 판단 | 유지·수정·채택·보류 이유 |
| 미검증·다음 변경 | 아직 실행하지 않은 조건과 다음 비교 |

학생 정보나 원문 입력은 공개 저장소에 올리지 않는다. 실제 실행하지 않은 조건은 미측정으로 표시한다.

## 막혔을 때의 확인 순서

| 증상 | 확인·해결 |
|---|---|
| 커널 선택 창만 표시 | Python·Jupyter 확장 설치 후 `.venv`의 Python 선택 |
| `ModuleNotFoundError` | 커널 경로 확인 → 해당 `.venv`로 `python -m pip install -r requirements.txt` |
| 환경 확인에서 저장소를 찾지 못함 | 저장소 전체를 열고 원래 `week10/lab/notebooks` 위치에서 실행 |
| `NameError` | 앞 셀 실행 여부 확인 → Restart Kernel → Run All |
| `asyncio.run() cannot be called...` | Jupyter에서는 제공된 최상위 `await`·`async with`를 그대로 사용 |
| 셀에 출력이 없음 | `assert`만 있는 셀은 오류 없이 끝나면 통과. 왼쪽 실행 번호와 완료 표시 확인 |
| 예상한 422·502·503·504 | 입력 오류·검색 오류·준비 실패·시간 초과를 의도적으로 만든 단계인지 확인 |
| 예상과 다른 `AssertionError` | 바로 위 코드·출력 확인. 변경한 값을 복원하고 처음부터 다시 실행. assert를 지워 통과시키지 않음 |
| 실행 시간·ID가 예시와 다름 | 고정값 비교 대신 상태·부모/자식 관계·분모·검증 결과를 비교 |

실습에서 바꾼 내용은 자신의 사본에 기록한다. 공개 저장소에는 학생 정보·원문 질문·실제 키를 올리지 않는다.